# 14b — Mémoire paginée d’agent

**Navigation** : [Version Python](14b_Paginated_Memory-Python.ipynb) | [Index Texte](README.md) | [Précédent : mémoire persistante](14_Persistent_Memory.ipynb) | [Suivant : Tree of Thoughts](15_Tree_of_Thoughts_Search.ipynb)

**Objectifs** : comparer fenêtre glissante, troncature cumulative, politiques FIFO/LRU/importance et pagination ; suivre l'identité des éléments et les coûts modélisés ; comprendre quelles parties du stockage sont réellement bornées.

**Prérequis** : listes, files, dictionnaires et notions de coût algorithmique.

**Durée estimée** : 70 minutes.

Le carnet présente des structures pédagogiques CPU en C#/.NET 9 BCL. Aucun LLM ou service externe n'est appelé. Le compagnon Python garde les mêmes démonstrations et mesures.

## Le probleme de la memoire non bornable

Un agent conversationnel qui appelle un LLM sur chaque tour accumule des messages dans la fenetre de contexte du modele. Trois symptomes apparaissent des que la conversation dure :

1. **Cout** : chaque tour peut repayer le contexte déjà transmis, ce qui pèse sur le coût réel d’un service. Les mesures synthétiques plus loin utilisent toutefois des unités de caractères, pas des tokens facturés.
2. **Latence** : la latence d'un appel LLM depend du nombre de tokens d'entree. La conversation ralentit mecaniquement au fil de l'eau.
3. **Degradation de la qualite** : les modeles oublient le milieu de leur contexte (lost-in-the-middle), et les anciens details ne servent plus rien une fois la discussion passee.

La memoire **persistante** de `14_Persistent_Memory.ipynb` (vecteur + similarite cosinus) repond au probleme du **transfer cross-session** : les lecons survivent aux runs et sont retrocedees par similarite. Elle ne repond **pas** au probleme du **cout intra-session** : dans une seule conversation longue, l'historique continue de croitre jusqu'a exploser la fenetre.

Ce notebook introduit la memoire **paginee** : trois mecanismes qui cooperent pour borner la taille de la fenetre de contexte vue par le LLM a chaque tour, tout en preservant les informations anciennes sous une forme accessible a la demande.

## Plan

1. Definitions et vocabulaire : fenetre glissante, resume progressif, eviction, paging.
2. Fenetre glissante (rolling window) : ne garder que les N derniers echanges intacts.
3. Resume progressif : condenser les messages evicques en un resume cumulatif.
4. Politique d'eviction : FIFO, LRU, importance-scored.
5. Memory paging (style Generative Agents / MemGPT) : hierarchie core/archival + retrieval a la demande.
6. Mesure : comparer un cout modélisé en caractères et le nombre de messages retenus sur une longue conversation synthetique, avec et sans pagination.
7. Trois exercices : fenetre parametrique, eviction LRU sur la memoire vectorielle de NB-14, transfer cross-paging multi-domaines.


## 0. Mise en place — structures BCL pures, sans service externe

Ce carnet s’exécute sur CPU sans endpoint, SDK agent ni modèle. Les résumés sont simulés par troncature et concaténation ; ce mécanisme n’est pas un résumé sémantique.


In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Runtime.CompilerServices;
using System.Text;
using System.Text.Json;
record Message(string Role, string Content, double Ts = 0, double Importance = 1);

class RollingWindow {
    public int Capacity { get; }
    private readonly Queue<Message> queue = new();
    public int Evicted { get; private set; }
    public int Count => queue.Count;
    public RollingWindow(int capacity) {
        if (capacity < 1) throw new ArgumentOutOfRangeException(nameof(capacity));
        Capacity = capacity;
    }
    public Message? Append(Message message) {
        Message? evicted = null;
        if (queue.Count == Capacity) { evicted = queue.Dequeue(); Evicted++; }
        queue.Enqueue(message);
        return evicted;
    }
    public string Render() => string.Join("\n", queue.Select(m => $"[{m.Role}] {m.Content}"));
}

class ProgressiveSummary {
    public List<string> Chunks { get; } = new();
    public int MaxCharsPerChunk { get; }
    public int StoredChars => Chunks.Sum(chunk => chunk.Length);
    public ProgressiveSummary(int maxCharsPerChunk = 80) {
        if (maxCharsPerChunk < 4) throw new ArgumentOutOfRangeException(nameof(maxCharsPerChunk));
        MaxCharsPerChunk = maxCharsPerChunk;
    }
    public void Add(Message message) {
        var content = message.Content.Length > MaxCharsPerChunk
            ? message.Content[..(MaxCharsPerChunk - 3)] + "..." : message.Content;
        Chunks.Add($"[{message.Role}] {content}");
    }
    public string Render(int? limit = null) {
        var body = string.Join("\n--\n", Chunks);
        if (limit is not int n) return body;
        if (n < 0) throw new ArgumentOutOfRangeException(nameof(limit));
        if (body.Length <= n) return body;
        if (n < 3) return new string('.', n);
        return body[..(n - 3)] + "...";
    }
}

class LRUMemory {
    private readonly int capacity;
    private readonly LinkedList<Message> items = new();
    private readonly Dictionary<Message, LinkedListNode<Message>> index = new(ReferenceEqualityComparer.Instance);
    public List<Message> Evicted { get; } = new();
    public int Count => items.Count;
    public LRUMemory(int capacity) {
        if (capacity < 1) throw new ArgumentOutOfRangeException(nameof(capacity));
        this.capacity = capacity;
    }
    public Message? Append(Message message) {
        if (index.ContainsKey(message)) return null;
        Message? evicted = null;
        if (items.Count == capacity) {
            evicted = items.First!.Value;
            items.RemoveFirst(); index.Remove(evicted); Evicted.Add(evicted);
        }
        index[message] = items.AddLast(message);
        return evicted;
    }
    public bool Touch(Message message) {
        if (!index.TryGetValue(message, out var node)) return false;
        items.Remove(node); items.AddLast(node); return true;
    }
    public string Render() => string.Join("\n", items.Select(m => $"[{m.Role}] {m.Content}"));
}

class ImportanceMemory {
    private readonly int capacity;
    private readonly List<Message> items = new();
    public ImportanceMemory(int capacity) {
        if (capacity < 1) throw new ArgumentOutOfRangeException(nameof(capacity));
        this.capacity = capacity;
    }
    public Message? Append(Message message) {
        Message? evicted = null;
        if (items.Count == capacity) {
            evicted = items.OrderBy(item => item.Importance).ThenBy(item => item.Ts).First();
            items.Remove(evicted);
        }
        items.Add(message); return evicted;
    }
    public string Render() => string.Join("\n", items.Select(m => $"[{m.Role}] (imp={m.Importance:F2}) {m.Content}"));
}

record Page(string Title, List<Message> Messages) {
    public string Render() => $"=== Page: {Title} ({Messages.Count} msg) ===\n" +
        string.Join("\n", Messages.Select(m => $"  [{m.Role}] {m.Content}"));
}

class PagedMemory {
    public List<Page> Pages { get; } = new();
    public int Current { get; private set; }
    public int AddPage(Page page) { Pages.Add(page); return Pages.Count - 1; }
    public Page SwapTo(int index) {
        if (index < 0 || index >= Pages.Count) throw new IndexOutOfRangeException($"page index {index} hors limites");
        Current = index; return Pages[index];
    }
    public Page CurrentPage() => Pages[Current];
    public int TotalMessages() => Pages.Sum(page => page.Messages.Count);
}
Console.WriteLine($"Runtime .NET {Environment.Version} ; structures BCL chargées : fenêtre, résumé, LRU par identité, importance, pages.");

Runtime .NET 10.0.14 ; structures BCL chargées : fenêtre, résumé, LRU par identité, importance, pages.



(18,19): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

(60,19): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

(84,19): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

(85,16): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

(62,16): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

(19,16): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



## 1. Definitions et vocabulaire

On formalise les **trois mecanismes** de pagination avant de les implementer. Le vocabulaire suit la litterature Generative Agents (Park et al., 2023) et MemGPT (Packer et al., 2023).

| Mecanisme | Question qu'il resout | Granularite |
|---|---|---|
| **Fenetre glissante** (rolling window) | Combien de messages recents garder intacts ? | Message (1 tour = 1 echange user/assistant) |
| **Resume progressif** (progressive summarization) | Que faire des messages evicques ? | Bloc resume cumulatif |
| **Eviction** (eviction policy) | Lequel supprimer en priorite quand la memoire est pleine ? | FIFO / LRU / importance-scored |
| **Memory paging** (core/archival retrieval) | Comment retrouver une information ancienne sans tout recharger ? | Page (sous-unite de memoire adressable) |

Un agent **reel** combine les quatre. Ici on les presente separement pour la clarte pedagogique, puis on les assemble dans une classe `PagedMemory`.

## 2. Fenetre glissante (rolling window)

Le mecanisme le plus simple : la memoire courante est un `deque(maxlen=N)` qui conserve les **N derniers messages**. Au-dela, les plus anciens sont automatiquement evicques (FIFO).

**Avantage** : O(1) en insert et en eviction, deterministe.
**Limite** : aucune information n'est preservee au-dela de la fenetre (oubli total). Un message important du tour 3 est perdu au tour N+3.

In [2]:
var rw=new RollingWindow(4);
for(int i=0;i<6;i++){var ev=rw.Append(new Message(i%2==0?"user":"assistant",$"message #{i}"));Console.WriteLine($"append #{i}: taille={rw.Count}"+(ev is null?"":$" -> évincé {ev.Content}"));}
Console.WriteLine("Fenêtre finale :\n"+rw.Render());Console.WriteLine($"Total évincés : {rw.Evicted}");

append #0: taille=1


append #1: taille=2


append #2: taille=3


append #3: taille=4


append #4: taille=4 -> évincé message #0


append #5: taille=4 -> évincé message #1


Fenêtre finale :
[user] message #2
[assistant] message #3
[user] message #4
[assistant] message #5


Total évincés : 2


**Lecture de la sortie.** Les messages #0 et #1 ont ete evicques (sortie `evince(s) : ['message #0']` puis `['message #1']` apparaissent sur les appends #4 et #5). Au final, la fenetre contient les messages #2 a #5, dans l'ordre. Le total evicque cumule est 2, et la taille de la fenetre reste bornee a 4 quel que soit le nombre d'appels.

## 3. Resume progressif

Quand un message est evince de la fenetre glissante, son contenu est **perdu**. Pour preserver l'information ancienne sous une forme compacte, on accumule un **resume cumulatif** : a chaque eviction, on concatene le message evince au resume en cours (ou on appelle un LLM pour le paraphraser).

**Strategie simplifiee** (notebook CPU-pur) : troncature a 80 caracteres par message evince, concatenation par saut de ligne. En production, remplacer par `summarize(msg) -> str` (cf `# HOOK`). Seul le rendu peut etre borne ; les chunks restent stockes en O(N), faute de consolidation.

In [3]:
var rw2=new RollingWindow(3);var sm=new ProgressiveSummary();
for(int i=0;i<7;i++){var m=new Message(i%2==0?"user":"assistant",$"Le projet X en est à l'itération {i} et les tests passent presque tous sauf un edge case.");var e=rw2.Append(m);if(e is not null)sm.Add(e);}
Console.WriteLine("Fenêtre finale :\n"+rw2.Render());Console.WriteLine($"Résumé tronqué ({sm.StoredChars} caractères stockés, {sm.Chunks.Count} chunks) :\n"+sm.Render(300));Console.WriteLine($"Stockage interne O(N) : chunks conservés={sm.Chunks.Count}; longueur rendue={sm.Render(300).Length} (plafond 300).");Console.WriteLine($"Limite de rendu 2 caractères : '{sm.Render(2)}' (longueur={sm.Render(2).Length}).");
Console.WriteLine($"Limite de rendu 2 caractères respecte le plafond : {sm.Render(2).Length==2}.");

Fenêtre finale :
[user] Le projet X en est à l'itération 4 et les tests passent presque tous sauf un edge case.
[assistant] Le projet X en est à l'itération 5 et les tests passent presque tous sauf un edge case.
[user] Le projet X en est à l'itération 6 et les tests passent presque tous sauf un edge case.


Résumé tronqué (358 caractères stockés, 4 chunks) :
[user] Le projet X en est à l'itération 0 et les tests passent presque tous sauf un ...
--
[assistant] Le projet X en est à l'itération 1 et les tests passent presque tous sauf un ...
--
[user] Le projet X en est à l'itération 2 et les tests passent presque tous sauf un ...
--
[assistant] Le proj...


Stockage interne O(N) : chunks conservés=4; longueur rendue=300 (plafond 300).


Limite de rendu 2 caractères : '..' (longueur=2).


Limite de rendu 2 caractères respecte le plafond : True.


**Lecture de la sortie.** Sur 7 messages avec une fenêtre de 3, 4 messages ont été évincés. Chacun est tronqué à 80 caractères puis ajouté aux chunks conservés ; la cellule limite seulement le rendu à 300 caractères. Les chunks ne sont pas consolidés et leur stockage croît avec le nombre d’évictions.


<a id="exercice-1"></a>

### Exercice 1 — coût marginal et fenêtres admissibles

Implemente une fonction `cout_marginal(window_size, n_turns, msg_len)` qui calcule le cout total d'une fenetre glissante sur `n_turns` messages, avec une taille de fenetre `window_size` et une longueur moyenne `msg_len`. Convention précise : la fenêtre est payée pleine dès le premier tour, donc coûtFenêtre=N×W×L ; coûtNonBorné=L×N(N+1)/2. Le seuil strict coûtFenêtre < coûtNonBorné/2 équivaut à W<(N+1)/4. Le minimum admissible vaut 1 quand il en existe un ; calcule aussi la plus grande fenêtre admissible et traite N=1, où aucune fenêtre entière ne satisfait le seuil.

**Indice** : le cout par tour d'une fenetre glissante est constant = `window_size * msg_len` des qu'on a fait `window_size` insertions. Le cout par tour d'une memoire non bornee est lineaire : `i * msg_len` au tour i. Trace les deux courbes et trouve leur intersection.

In [4]:
record FenetreResult(double CoutTotal, bool SeuilAtteint, int FenetreMin, int FenetreMax);
FenetreResult? CoutMarginalEtudiant(int windowSize, int nTurns, int msgLen) { return null; } // TODO etudiant
long CoutFenetreOracle(int w,int n,int len)=>(long)n*w*len;
long CoutNonBorneOracle(int n,int len)=>(long)len*n*(n+1)/2;
FenetreResult FenetreOracle(int w,int n,int len){bool feasible=1<=w&&w<=n&&w<(n+1)/4.0;int max=Enumerable.Range(1,n).Where(candidate=>candidate<(n+1)/4.0).DefaultIfEmpty(0).Max();return new FenetreResult(CoutFenetreOracle(w,n,len),feasible,max>0?1:0,max);}
bool AcceptFenetre(FenetreResult? actual,int w,int n,int len){if(actual is not FenetreResult value)return false;var expected=FenetreOracle(w,n,len);return double.IsFinite(value.CoutTotal)&&Math.Abs(value.CoutTotal-expected.CoutTotal)<1e-9&&value.SeuilAtteint==expected.SeuilAtteint&&value.FenetreMin==expected.FenetreMin&&value.FenetreMax==expected.FenetreMax;}
var fenetreFixtures=new[]{(W:1,N:8,L:20,Max:2),(W:2,N:8,L:20,Max:2),(W:3,N:12,L:10,Max:3),(W:10,N:40,L:50,Max:10),(W:1,N:1,L:20,Max:0)};
bool fenetreAnswerPresent=CoutMarginalEtudiant(10,100,120) is not null;
bool fenetreFixturesPass=fenetreFixtures.All(f=>AcceptFenetre(CoutMarginalEtudiant(f.W,f.N,f.L),f.W,f.N,f.L)&&FenetreOracle(f.W,f.N,f.L).FenetreMax==f.Max);
bool fenetreMutantsRejected=fenetreFixtures.All(f=>!AcceptFenetre(new FenetreResult(0,false,0,0),f.W,f.N,f.L)&&!AcceptFenetre(new FenetreResult(double.NaN,false,1,1),f.W,f.N,f.L));
Console.WriteLine($"Exercice 1 : {(fenetreAnswerPresent?"réponse présente":"à compléter")}; fixtures={fenetreFixtures.Length}; valeurs oracle N=8/12/40/1 → max=2/3/10/0; contrat réel respecté={fenetreFixturesPass}; mutants constants/NaN rejetés={fenetreMutantsRejected}.");
Console.WriteLine("Convention : coût fenêtre=N×W×L dès le premier tour ; coût non borné=L×N(N+1)/2. Seuil strict : W<(N+1)/4 ; min=1 si admissible, sinon aucune fenêtre (bornes 0/0).");

Exercice 1 : à compléter; fixtures=5; valeurs oracle N=8/12/40/1 → max=2/3/10/0; contrat réel respecté=False; mutants constants/NaN rejetés=True.


Convention : coût fenêtre=N×W×L dès le premier tour ; coût non borné=L×N(N+1)/2. Seuil strict : W<(N+1)/4 ; min=1 si admissible, sinon aucune fenêtre (bornes 0/0).



(2,14): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

(6,33): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



**Lecture de l’exercice 1.** Le stub reste à compléter. Le contrat distingue fenêtre minimale, fenêtre maximale et ensemble admissible ; les sorties de garde ne prétendent pas mesurer une courbe étudiante.


## 4. Politique d'eviction -- FIFO, LRU, importance-scored

La fenetre glissante est FIFO par construction. Mais on peut choisir une **politique d'eviction** : quand la memoire est pleine, **lequel** sacrifier ?

- **FIFO** : le plus ancien (defaut de `deque(maxlen=N)`).
- **LRU** (Least Recently Used) : celui qu'on n'a pas lu/cite depuis le plus longtemps. Necessite un compteur d'acces par message.
- **Importance-scored** : celui dont l'importance est la plus basse. Necessite un score (calcule par le LLM, par l'usage, par les outils appeles, etc.).

Implementation : `LRUMemory` et `ImportanceMemory` comme variantes de `RollingWindow` (memes operations, politique de remplacement differente).

In [5]:
var lru=new LRUMemory(3);
var msgs=new List<Message>();
for(int i=0;i<5;i++){
    var message=new Message("user",$"message #{i}",i);
    msgs.Add(message); lru.Append(message);
}
Console.WriteLine("Après 5 insertions (ordre LRU vers MRU) :\n"+lru.Render());
bool absentTouch=lru.Touch(msgs[1]);
Console.WriteLine($"Touch de #1 déjà évincé réussi : {absentTouch} ; contenu inchangé :\n{lru.Render()}");
bool presentTouch=lru.Touch(msgs[2]);
Console.WriteLine($"Touch du LRU #2 courant réussi : {presentTouch} ; ordre après promotion :\n{lru.Render()}");
var evicted=lru.Append(new Message("user","message #5",5));
Console.WriteLine($"Insertion #5 évince {evicted?.Content} ; contenu final :\n{lru.Render()}");
var importanceMemory=new ImportanceMemory(3);
foreach(var item in new[]{("a",1.0,1.0),("b",1.0,2.0),("c",1.0,3.0),("d",1.0,4.0),("PENSEE-CLE : ne jamais committer un secret",10.0,5.0)})
    importanceMemory.Append(new Message("user",item.Item1,item.Item3,item.Item2));
Console.WriteLine("Importance après 5 insertions :\n"+importanceMemory.Render());
var tied=new ImportanceMemory(2);
tied.Append(new Message("user","égalité première",1,1));
tied.Append(new Message("user","égalité seconde",1,1));
var tieVictim=tied.Append(new Message("user","nouvelle",2,2));
bool stableTimestampTie=tieVictim?.Content=="égalité première";
Console.WriteLine($"Horodatages égaux : éviction stable du premier élément = {stableTimestampTie}.");
var identityProbe=new LRUMemory(2);
var original=new Message("user","même valeur",10);
identityProbe.Append(original);
var duplicateEviction=identityProbe.Append(original);
var equalDistinct=new Message("user","même valeur",10);
identityProbe.Append(equalDistinct);
var thirdIdentity=new Message("user","troisième",11);
var identityVictim=identityProbe.Append(thirdIdentity);
bool identityContract=identityProbe.Count==2&&duplicateEviction is null&&object.ReferenceEquals(identityVictim,original)&&!identityProbe.Touch(original)&&identityProbe.Touch(equalDistinct);
Console.WriteLine($"LRU par identité : même instance unique, valeur égale distincte, Touch/éviction par référence = {identityContract}.");

Après 5 insertions (ordre LRU vers MRU) :
[user] message #2
[user] message #3
[user] message #4


Touch de #1 déjà évincé réussi : False ; contenu inchangé :
[user] message #2
[user] message #3
[user] message #4


Touch du LRU #2 courant réussi : True ; ordre après promotion :
[user] message #3
[user] message #4
[user] message #2


Insertion #5 évince message #3 ; contenu final :
[user] message #4
[user] message #2
[user] message #5


Importance après 5 insertions :
[user] (imp=1.00) c
[user] (imp=1.00) d
[user] (imp=10.00) PENSEE-CLE : ne jamais committer un secret


Horodatages égaux : éviction stable du premier élément = True.


LRU par identité : même instance unique, valeur égale distincte, Touch/éviction par référence = True.


**Lecture de la sortie.** Après cinq insertions dans une capacité de trois, la LRU contient #2/#3/#4. Toucher #1 évincé ne le ressuscite pas ; toucher #2 le rend le plus récent. L’insertion #5 évince alors #3. Dans l’exemple d’importance, le message noté 10 survit aux entrées notées 1.


<a id="exercice-2"></a>

### Exercice 2 — mémoire vectorielle, récence et importance

Reprends `MemoireVectorielle` du notebook 14 (`MyIA.AI.Notebooks/GenAI/Texte/14_Persistent_Memory.ipynb`) et ajoute-lui une politique d'éviction **importance + récence** : à chaque `ajouter`, si `len(entrees) > max_entrees`, évince l'entrée de score pondéré le plus faible. La recence est mesuree par l'age en secondes. L'importance est la **longueur de la lecon** (proxy : une lecon detaillee est plus importante qu'un acquittement sec, comme demontre dans NB-14).

**Indice** : le `score` peut etre `0.5 * (1 / (1 + age_en_secondes)) + 0.5 * (len(lecon) / max_lecon_len)`. A chaque ajout, recalcule les scores et evince le plus bas. Verifie que la lecon correcte (`PENSEE-CLE : ne jamais committer un secret`) reste en memoire apres plusieurs insertions.

In [6]:
record VectorEntry(string Spec,string Lesson,double Timestamp,double[] Vector);
class MemoireVectoriellePagineeEtudiante {
    public List<VectorEntry> Entries { get; } = new();
    public int Capacity { get; }
    public MemoireVectoriellePagineeEtudiante(int capacity=3) {
        if (capacity < 1) throw new ArgumentOutOfRangeException(nameof(capacity));
        Capacity=capacity;
    }
    public void Ajouter(string spec,string lesson,double timestamp,Func<string,double[]> embedder) { /* TODO etudiant */ }
    public double? Score(VectorEntry entry,double now,double maxLessonLength) { return null; } // TODO etudiant
    public List<string> Retroceder(string query,int k,Func<string,double[]> embedder) { return new(); } // TODO etudiant
}
double[] EmbedDeterministe(string text) => text.Contains("code",StringComparison.OrdinalIgnoreCase)
    ? new[]{1.0,0.0}
    : text.Contains("préférence",StringComparison.OrdinalIgnoreCase) ? new[]{0.0,1.0} : new[]{1.0,1.0};
double ExpectedScore(string lesson,double timestamp,double now,double maxLength) =>
    0.5/(1+now-timestamp)+0.5*lesson.Length/maxLength;
double Cosine(double[] left,double[] right) {
    if(left.Length!=right.Length||left.Length==0)return double.NaN;
    double dot=left.Zip(right,(a,b)=>a*b).Sum();
    double ln=Math.Sqrt(left.Sum(x=>x*x)),rn=Math.Sqrt(right.Sum(x=>x*x));
    return ln==0||rn==0?double.NaN:dot/(ln*rn);
}
var vectorMemory=new MemoireVectoriellePagineeEtudiante(3);
var vectorFixtures=new[]{("code ancien",10.0),("préférence concise",90.0),("code récent",95.0),("code détail prioritaire",99.0)};
foreach(var fixture in vectorFixtures) vectorMemory.Ajouter("mémoire",fixture.Item1,fixture.Item2,EmbedDeterministe);
double nowFixture=100.0,maxLessonLength=vectorFixtures.Max(x=>x.Item1.Length);
var expectedKept=vectorFixtures.OrderByDescending(x=>ExpectedScore(x.Item1,x.Item2,nowFixture,maxLessonLength)).ThenByDescending(x=>x.Item2).Take(3).Select(x=>x.Item1).OrderBy(x=>x).ToArray();
var actualKept=vectorMemory.Entries.Select(x=>x.Lesson).OrderBy(x=>x).ToArray();
bool addEvictionPass=vectorMemory.Entries.Count==vectorMemory.Capacity&&actualKept.SequenceEqual(expectedKept)&&vectorMemory.Entries.All(e=>e.Vector.Length==2&&e.Vector.All(double.IsFinite));
bool scorePass=vectorMemory.Entries.Count==3&&vectorMemory.Entries.All(entry=>{var actual=vectorMemory.Score(entry,nowFixture,maxLessonLength);return actual is double value&&double.IsFinite(value)&&value>=0&&value<=1&&Math.Abs(value-ExpectedScore(entry.Lesson,entry.Timestamp,nowFixture,maxLessonLength))<1e-9;});
var queryVector=EmbedDeterministe("code");
var expectedRetrieval=vectorMemory.Entries.OrderByDescending(entry=>Cosine(queryVector,entry.Vector)).ThenBy(entry=>entry.Lesson,StringComparer.Ordinal).Take(2).Select(entry=>entry.Lesson).ToArray();
var actualRetrieval=vectorMemory.Retroceder("code",2,EmbedDeterministe);
bool retrievalPass=actualRetrieval.Count==2&&actualRetrieval.SequenceEqual(expectedRetrieval);
bool ValidScore(double? actual,double expected)=>actual is double value&&double.IsFinite(value)&&value>=0&&value<=1&&Math.Abs(value-expected)<1e-9;
bool mutantsRejected=vectorFixtures.All(fixture=>{double expected=ExpectedScore(fixture.Item1,fixture.Item2,nowFixture,maxLessonLength);return !ValidScore(0.5,expected)&&!ValidScore(double.NaN,expected);});
bool scoreNull=vectorMemory.Score(new VectorEntry("x","x",1,new[]{1.0,0.0}),nowFixture,1) is null;
Console.WriteLine($"Exercice 2 : {(vectorMemory.Entries.Count==0?"à compléter":"réponse présente")}; capacité/cardinalité/éviction/vector dimension={addEvictionPass}; scores réels sur {vectorMemory.Entries.Count} entrées={scorePass}; top-2 exact={retrievalPass}; constante/NaN et stub null détectés={mutantsRejected&&scoreNull}.");
Console.WriteLine("Callbacks embedding/horloge déterministes. Score oracle = 0,5/(1+âge) + 0,5×longueur(leçon)/longueur max ; les sorties de garde n'implémentent pas le stub.");

Exercice 2 : à compléter; capacité/cardinalité/éviction/vector dimension=False; scores réels sur 0 entrées=False; top-2 exact=False; constante/NaN et stub null détectés=True.


Callbacks embedding/horloge déterministes. Score oracle = 0,5/(1+âge) + 0,5×longueur(leçon)/longueur max ; les sorties de garde n'implémentent pas le stub.


**Lecture de l’exercice 2.** Le helper reste non résolu. Les fixtures emploient callbacks déterministes pour rendre le score vérifiable ; le score combine récence et importance par moyenne pondérée.


## 5. Memory paging (style Generative Agents / MemGPT)

Le **memory paging** est le mecanisme des systemes type MemGPT (Packer et al., 2023) et Generative Agents (Park et al., 2023) : la memoire est decoupee en **pages** (sous-unites adressables), et l'agent ne voit a chaque tour que la **page courante** dans son contexte. Pour acceder a une page non-courante, il **appelle un outil** de retrieval qui swap-in la page demandee.

L'analogie est volontairement systeme d'exploitation : la memoire de l'agent est un espace d'adressage virtuel, le contexte du LLM est la memoire physique (RAM), et le swap est un outil de retrieval.

Implementation simplifiee ici : `PagedMemory` tient une liste de pages ; `current_page` est celle injectee dans le prompt. `swap_to(idx)` charge une autre page (coût symbolique simulé).

In [7]:
var pm=new PagedMemory();pm.AddPage(new Page("onboarding",new(){new("user","Bonjour, je m'appelle Alice."),new("assistant","Bonjour Alice, bienvenue.")}));pm.AddPage(new Page("bug-investigation",new(){new("user","Le test #42 échoue."),new("assistant","Voyons la trace, ligne 17."),new("user","C'était bien la ligne 17, merci.")}));pm.AddPage(new Page("preferences",new(){new("user","Je préfère les réponses en français."),new("assistant","Compris, je répondrai en français.")}));
Console.WriteLine($"Mémoire paginée : {pm.Pages.Count} pages, {pm.TotalMessages()} messages ; page courante #{pm.Current} {pm.CurrentPage().Title}.");foreach(var (p,i) in pm.Pages.Select((p,i)=>(p,i)))Console.WriteLine($"Page #{i}: {p.Title}" );var loaded=pm.SwapTo(2);Console.WriteLine($"Swap index 2 :\n{loaded.Render()}");bool invalidRejected=false;try{pm.SwapTo(3);}catch(IndexOutOfRangeException){invalidRejected=true;}Console.WriteLine($"Index de page invalide rejeté : {invalidRejected}; stockage de pages et archives reste O(N).");

Mémoire paginée : 3 pages, 7 messages ; page courante #0 onboarding.


Page #0: onboarding


Page #1: bug-investigation


Page #2: preferences


Swap index 2 :
=== Page: preferences (2 msg) ===
  [user] Je préfère les réponses en français.
  [assistant] Compris, je répondrai en français.


Index de page invalide rejeté : True; stockage de pages et archives reste O(N).


**Lecture de la sortie.** La démonstration contient 3 pages et 7 messages. Le swap charge `preferences` ; seul le contenu de cette page est rendu. Les autres pages restent stockées en mémoire, et le swap n’est pas facturé dans le calcul synthétique.


<a id="exercice-3"></a>

### Exercice 3 — navigation multi-domaines par dernière page vue

Etends `PagedMemory` pour gerer des **domaines** (par exemple : `code`, `math`, `preferences`). Chaque page porte un `domaine`, et la methode `swap_to_domain(dom)` charge la **dernière page vue** de ce domaine (au lieu de demander un index numerique). Teste : un utilisateur parle de code pendant 10 tours, puis de preferences pendant 5 tours, puis revient au code -- le swap-to-domain re-charge-t-il bien la derniere page code vue ?

**Indice** : ajoute un champ `domaine: str` a `Page`, et un `Dict[str, int]` `last_seen[dom] -> index de page réellement visitée` dans `PagedMemory`. La methode `swap_to_domain(dom)` consulte `last_seen[dom]` et appelle `swap_to(...)`.

In [8]:
class PagedMemoryMultiDomaineEtudiante {
    public List<(string Domaine,string Contenu)> Pages { get; } = new();
    public Dictionary<string,int> LastSeen { get; } = new();
    public int Current { get; private set; }
    public int AddPage(string domaine,string contenu) { /* TODO etudiant */ return -1; }
    public int SwapTo(int index) { /* TODO etudiant : enregistrer la visite */ return -1; }
    public int SwapToDomain(string domaine) { /* TODO etudiant : retrouver la dernière page visitée */ return -1; }
}
var multiFixture=new PagedMemoryMultiDomaineEtudiante();
int oldCodeIndex=multiFixture.AddPage("code","page-code-ancienne");
int visitedOldCode=multiFixture.SwapTo(oldCodeIndex);
int latestCodeIndex=multiFixture.AddPage("code","page-code-récente");
int preferencesIndex=multiFixture.AddPage("preferences","page-preferences");
int visitedPreferencesIndex=multiFixture.SwapTo(preferencesIndex);
int returnedCodeIndex=multiFixture.SwapToDomain("code");
int expectedOldCodeIndex=0;int expectedLatestCodeIndex=2;
bool lastSeenAccepted=oldCodeIndex==expectedOldCodeIndex&&latestCodeIndex==expectedLatestCodeIndex&&visitedOldCode==expectedOldCodeIndex&&visitedPreferencesIndex==1&&returnedCodeIndex==expectedOldCodeIndex&&returnedCodeIndex!=expectedLatestCodeIndex;
bool missingDomainRejected=false;try{_ = multiFixture.SwapToDomain("absent");}catch(InvalidOperationException){missingDomainRejected=true;}
bool multiStubIncomplete=oldCodeIndex<0||latestCodeIndex<0||returnedCodeIndex<0;
Console.WriteLine($"Exercice 3 : {(multiStubIncomplete?"à compléter":"réponse présente")}; fixture visite code ancien, ajoute code récent, visite préférences puis revient au code; last-seen correct={lastSeenAccepted}; index dernière insertion incorrect={expectedLatestCodeIndex}; domaine absent rejeté={missingDomainRejected}.");

Exercice 3 : à compléter; fixture visite code ancien, ajoute code récent, visite préférences puis revient au code; last-seen correct=False; index dernière insertion incorrect=2; domaine absent rejeté=False.


**Lecture de l’exercice 3.** La fixture revisite le domaine `code` après `preferences` : elle distingue l’index de dernière page vue d’un simple index ajouté en dernier.


## 6. Mesures synthétiques — unités de coût en caractères

Le calcul ci-dessous compare deux profils (100 et 1000 tours) à `MSG_AVG_LEN=120`, `WINDOW=10`, `RESUME_MAX=200` et `PAGE_SIZE=5`. Il s’agit d’un proxy déterministe en caractères, pas d’un tokenizer ni d’une facture d’API.


In [9]:
const int MSG_AVG_LEN=120, WINDOW=10, RESUME_MAX=200, PAGE_SIZE=5;
(long Unbounded,long Window,long Paged) Gains(int n){long unbounded=0,window=0,paged=0;var rw=new RollingWindow(WINDOW);var sm=new ProgressiveSummary(40);for(int i=0;i<n;i++){unbounded+=(long)MSG_AVG_LEN*(i+1);var e=rw.Append(new Message(i%2==0?"user":"assistant",$"Message #{i:000} - "+new string('x',MSG_AVG_LEN)));if(e is not null)sm.Add(e);window+=(long)WINDOW*MSG_AVG_LEN+Math.Min(RESUME_MAX,sm.StoredChars);paged+=(long)PAGE_SIZE*MSG_AVG_LEN+RESUME_MAX;}return (unbounded,window,paged);}
var profils=new[]{100,1000};Console.WriteLine($"Profil synthétique : chars/message={MSG_AVG_LEN}, W={WINDOW}, P={PAGE_SIZE}, résumé={RESUME_MAX} (unités de coût modélisées, pas jetons mesurés).");foreach(var n in profils){var g=Gains(n);Console.WriteLine($"{n} tours : non borné={g.Unbounded:N0}; fenêtre+résumé={g.Window:N0} (x{(double)g.Unbounded/g.Window:F1}); paging={g.Paged:N0} (x{(double)g.Unbounded/g.Paged:F1})");}
Console.WriteLine("Modèle : fenêtre payée pleine dès le premier tour ; préfixes/rôles omis ; swap gratuit. Rendu borné ≠ stockage borné : pages et chunks archivés croissent en O(N).");

Profil synthétique : chars/message=120, W=10, P=5, résumé=200 (unités de coût modélisées, pas jetons mesurés).


100 tours : non borné=606,000; fenêtre+résumé=137,690 (x4.4); paging=80,000 (x7.6)


1000 tours : non borné=60,060,000; fenêtre+résumé=1,397,690 (x43.0); paging=800,000 (x75.1)


Modèle : fenêtre payée pleine dès le premier tour ; préfixes/rôles omis ; swap gratuit. Rendu borné ≠ stockage borné : pages et chunks archivés croissent en O(N).


**Lecture des mesures.** À 100 tours, les sorties calculent 606 000 unités non bornées, 137 690 pour fenêtre + résumé (facteur 4,4) et 80 000 pour paging (7,6). À 1000 tours, elles donnent 60 060 000, 1 397 690 (43,0) et 800 000 (75,1). Ces nombres suivent les conventions du modèle (W×120 dès le début, P×120+200 par tour, sans préfixes ni swap facturé) ; ils ne représentent ni des tokens réels ni une économie mesurée sur un service. La fenêtre affichée est plafonnée, mais les chunks du résumé et les pages restent stockés en O(N).


## 7. À retenir

La fenêtre glissante borne le nombre de messages affichés, le résumé conserve une trace tronquée des évictions et le paging rend une page ancienne adressable. La LRU dépend des accès effectifs ; une consultation d’un élément évincé ne le réinsère pas. La politique importance de l’exemple retire d’abord l’entrée la moins importante, puis la plus ancienne en cas d’égalité.

### Limites des démonstrations

- Le résumé est une **troncature**, pas une synthèse sémantique. Les chunks conservés en mémoire et les archives de pages croissent en O(N) ; seul le texte rendu peut être plafonné.
- Les valeurs appelées « coûts » sont des unités modélisées à partir de caractères, sans tokenizer ni facturation réelle. Le modèle paie W×120 dès le premier tour et P×120+200 à chaque tour, sans compter préfixes/rôles ; le swap est gratuit dans la formule. Les valeurs ne sont donc pas une mesure réelle du contexte d’un modèle.
- Les profils de 100 et 1000 tours donnent des résultats du calcul synthétique à W=10, P=5, longueur moyenne 120 et budget de résumé 200 ; ils n’incluent pas le coût d’un LLM.

### Récapitulatif des trois exercices

1. [Coût marginal paramétrique](#exercice-1) : garder le stub non résolu, mesurer la plus petite et la plus grande fenêtre admissibles, et expliquer le cas où aucun intervalle n’existe.
2. [Mémoire vectorielle](#exercice-2) : combiner récence et importance par un score pondéré déterministe, avec callbacks d’horloge et d’embedding contrôlés.
3. [Pagination par domaine](#exercice-3) : sélectionner la dernière page réellement vue, et non la dernière page ajoutée.

Les sorties de contrôle rejettent des valeurs non finies, des réponses constantes hors domaine et des dimensions erronées sur des fixtures bornées. Elles ne certifient pas n’importe quelle solution ni une courbe arbitraire.

## Vérifiez votre compréhension

<details><summary>Le résumé de la démonstration est-il sémantique ?</summary>

Non. Chaque morceau est tronqué à une longueur donnée, puis concaténé ; aucun modèle ne reformule le sens.

</details>

<details><summary>Pourquoi la mémoire complète reste-t-elle O(N) ?</summary>

La fenêtre visible est bornée, mais les objets Message, les chunks du résumé et les pages archivées restent stockés tant qu’ils ne sont pas supprimés ou consolidés.

</details>

<details><summary>Que mesure le calcul de coût ?</summary>

Une approximation pédagogique en caractères sous des conventions explicites, pas des tokens réellement envoyés ou facturés.

</details>

## Pour aller plus loin

- [Version Python de la démonstration](14b_Paginated_Memory-Python.ipynb)
- [Notebook 14 — mémoire persistante](14_Persistent_Memory.ipynb)
- [Notebook 12 — réflexivité](12_Reflexion.ipynb) et [13 — agents/outils](13_Agentic_Frameworks.ipynb)
- Park et al. (2023), *Generative Agents* ; Packer et al. (2023), *MemGPT*.
